In [ ]:
# =============================================================================
# STOCHASTIC ATTRACTION MAPS IN THE QUASI-CENTIPEDE GAME
# =============================================================================
# Multi-Agent Reinforcement Learning and Backward Induction
#
# Author: Peer-Review
# Revised: August 2026
# Purpose: Replicated finite-horizon comparison of three same-policy learning
#          processes in the unit-translated Smead three-stage quasi-centipede.
#
# Each policy is evaluated on a 50 x 50 slice through its initial-state space.
# The two axes vary the players' early-versus-late action bias. Each grid setting
# receives multiple independent stochastic trajectories. Trajectories advance
# through common horizons without resets and stop when their window-level action
# frequencies stabilize, subject to a 10,000,000-round ceiling.
#
# The primary displays estimate the probability of terminal concentration on
# each outcome at every initial-bias setting. They are stochastic attraction maps
# over prescribed two-dimensional slices, not complete mathematical basins over
# every possible policy state. The computations do not prove or validate an
# asymptotic convergence theorem.
#
# Policies:
#   1. Sample-mean epsilon-greedy with epsilon_t = 1/(t+1)
#   2. Gaussian Thompson Sampling with pseudo-variance 4
#   3. Repaired anytime Exp3 with vanishing gamma_t and eta_t
# =============================================================================


In [ ]:
# =============================================================================
# CELL 1: LIBRARY IMPORTS AND SETUP
# =============================================================================
import csv
import hashlib
import json
import os
import platform
import tempfile
from collections import deque
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path
from time import time

# Reduce BLAS threading nondeterminism (must be set before importing numpy).
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import seaborn as sns
import ray

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

BASE_SEED = 316
SEED_METHOD = (
    "A policy-specific SeedSequence(base_seed, policy_code, replicate_index), "
    "shared across grid settings as a common-random-number replicate bank, "
    "then split into independent private player generators"
)
POLICY_SEED_CODES = {'epsilon_greedy': 101, 'thompson': 211, 'exp3': 307}

print("Libraries loaded successfully.")
print(f"NumPy version: {np.__version__}")
print(f"BASE_SEED: {BASE_SEED}")


In [ ]:
# =============================================================================
# CELL 2: RAY INITIALIZATION
# =============================================================================
# Ray distributes one seeded continuous trajectory for each initial-bias grid
# setting. A 50 x 50 display therefore contains 2,500 settings, not repeated
# trials at each setting.

if ray.is_initialized():
    ray.shutdown()

ray.init(ignore_reinit_error=True)
print(f"Ray initialized with {ray.cluster_resources().get('CPU', 0):.0f} CPUs")

try:
    dashboard_url = ray.get_dashboard_url()  # type: ignore
    if dashboard_url:
        print(f"Ray dashboard: {dashboard_url}")
except Exception:
    pass


In [ ]:
# =============================================================================
# CELL 3: TWO-PLAYER GAME CLASS
# =============================================================================
class TwoPlayerGame:
    """Two-player repeated game with simultaneous action selection."""

    def __init__(self, payoff_matrix_p1, payoff_matrix_p2=None):
        self.payoff_matrix_p1 = np.array(payoff_matrix_p1, dtype=np.float64)
        if payoff_matrix_p2 is None:
            # In a symmetric game, P2's payoff at (a1,a2) is P1's payoff at
            # the transposed action profile (a2,a1).
            self.payoff_matrix_p2 = self.payoff_matrix_p1.T.copy()
        else:
            self.payoff_matrix_p2 = np.array(payoff_matrix_p2, dtype=np.float64)
        if self.payoff_matrix_p1.shape != self.payoff_matrix_p2.shape:
            raise ValueError("Player payoff matrices must have the same shape")
        if self.payoff_matrix_p1.ndim != 2 or self.payoff_matrix_p1.shape[0] != self.payoff_matrix_p1.shape[1]:
            raise ValueError("This experiment requires square payoff matrices")
        self.n_actions = self.payoff_matrix_p1.shape[0]

    def get_payoffs(self, action_p1, action_p2):
        return (
            self.payoff_matrix_p1[action_p1, action_p2],
            self.payoff_matrix_p2[action_p1, action_p2],
        )


print("TwoPlayerGame class defined.")


In [ ]:
# =============================================================================
# CELL 4: UNIT-TRANSLATED SMEAD QUASI-CENTIPEDE
# =============================================================================
# The notebook uses the three-stage specialization of
#
#   u_i(k,j) = 2k+5  if k<j,
#              2k+2  if k=j,
#              2j+1  if k>j.
#
# This is Smead's finite-N construction translated upward by one payoff unit in
# every cell. Smead did not print this +1 table. The uniform translation leaves
# payoff differences, dominance relations, Nash equilibria, and raw external-
# regret differences unchanged.
#
# The weak-dominance order is iterative: stage 1 weakly dominates stage 2 in
# the full three-stage game; after stage 2 is removed, stage 0 weakly dominates
# stage 1 in the reduced game. The unique Nash equilibrium is (0,0).

def create_quasi_centipede_game():
    """Return the unit-translated Smead three-stage quasi-centipede."""
    payoff_matrix_p1 = np.array([
        [2, 5, 5],
        [1, 4, 7],
        [1, 3, 6],
    ], dtype=np.float64)
    payoff_matrix_p2 = payoff_matrix_p1.T.copy()
    return TwoPlayerGame(payoff_matrix_p1, payoff_matrix_p2)


def display_quasi_centipede_info():
    print("Unit-translated Smead three-stage quasi-centipede")
    print("=" * 60)
    print("\nPayoff Matrix (P1 payoff, P2 payoff):")
    print("               P2: Take=0   P2: Take=1   P2: Take=2")
    print("P1: Take=0      (2, 2)       (5, 1)       (5, 1)")
    print("P1: Take=1      (1, 5)       (4, 4)       (7, 3)")
    print("P1: Take=2      (1, 5)       (3, 7)       (6, 6)")
    print("\nUnique Nash equilibrium: (Take=0, Take=0)")
    print("\nIterated weak dominance:")
    print("  1. Stage 1 weakly dominates stage 2 in the full game.")
    print("  2. After deleting stage 2, stage 0 weakly dominates stage 1.")
    print("=" * 60)


game = create_quasi_centipede_game()
display_quasi_centipede_info()


In [ ]:
# =============================================================================
# CELL 5: STATIC PAYOFF-CODE CHECK WHEN THE NOTEBOOK IS RUN
# =============================================================================
print("Checking the unit-translated payoff table...")
game = create_quasi_centipede_game()
expected = {
    (0, 0): (2, 2), (0, 1): (5, 1), (0, 2): (5, 1),
    (1, 0): (1, 5), (1, 1): (4, 4), (1, 2): (7, 3),
    (2, 0): (1, 5), (2, 1): (3, 7), (2, 2): (6, 6),
}

for (a1, a2), target in expected.items():
    observed = game.get_payoffs(a1, a2)
    if not np.allclose(observed, target):
        raise AssertionError(f"Payoff mismatch at {(a1, a2)}: {observed} != {target}")

print("Payoff table matches the notebook's unit-translated Smead normalization.")
print(f"Backward-induction profile payoff: {game.get_payoffs(0, 0)}")
print(f"Stage-2 diagonal payoff: {game.get_payoffs(2, 2)}")


In [ ]:
# =============================================================================
# CELL 6: SAMPLE-MEAN EPSILON-GREEDY POLICY
# =============================================================================
# At zero-indexed round t, epsilon_t = 1/(t+1). Exploration is uniform;
# exploitation uses uniform random tie-breaking over the current maximizers.
# Only the selected action is updated, using its exact sample mean.
#
# The grid initializes Q_0 = [-beta/2, 0, beta/2]. This preserves the original
# stage-2-versus-stage-0 value difference beta while making the direction plain:
# beta<0 favors stage 0, beta>0 favors stage 2, and stage 1 remains neutral.
# Only beta=0 is the exact zero-initialized process in Supplement v5.

class EpsilonGreedyPolicy:
    def __init__(self, arms: int, rng: np.random.Generator, initial_bias: float = 0.0):
        self.arms = arms
        self.rng = rng
        self.est_payoffs = np.zeros(arms, dtype=np.float64)
        self.K = np.zeros(arms, dtype=np.int64)
        self.timestep = 0
        if arms >= 3:
            self.est_payoffs[0] = -initial_bias / 2.0
            self.est_payoffs[2] = initial_bias / 2.0

    def policy_action(self) -> int:
        epsilon = 1.0 / (self.timestep + 1)
        if self.rng.random() > epsilon:
            max_q = np.max(self.est_payoffs)
            best_actions = np.flatnonzero(np.isclose(self.est_payoffs, max_q, rtol=0.0, atol=1e-10))
            return int(self.rng.choice(best_actions))
        return int(self.rng.integers(self.arms))

    def policy_update(self, action: int, reward: float) -> None:
        self.K[action] += 1
        self.est_payoffs[action] += (reward - self.est_payoffs[action]) / self.K[action]
        self.timestep += 1


print("EpsilonGreedyPolicy class defined.")


In [ ]:
# =============================================================================
# CELL 7: GAUSSIAN THOMPSON SAMPLING POLICY
# =============================================================================
# The experiment uses finite positive pseudo-likelihood variance sigma^2=4,
# supplied as reward_std=2.0. This is a numerical pseudo-variance, not a claim
# about the deterministic payoff table.
#
# The grid initializes prior means [-mu/2, 0, mu/2]. This preserves the original
# stage-2-versus-stage-0 mean difference mu. Negative values favor stage 0;
# positive values favor stage 2; stage 1 remains neutral.

class ThompsonSampling:
    def __init__(self, arms: int, rng: np.random.Generator, reward_std: float = 2.0,
                 prior_mean: float = 0.0, prior_std: float = 10.0,
                 initial_bias: float = 0.0):
        if reward_std <= 0 or prior_std <= 0:
            raise ValueError("Gaussian variances must be finite and positive")
        self.arms = arms
        self.rng = rng
        self.posterior_mean = np.full(arms, prior_mean, dtype=np.float64)
        self.posterior_var = np.full(arms, prior_std ** 2, dtype=np.float64)
        if arms >= 3:
            self.posterior_mean[0] = prior_mean - initial_bias / 2.0
            self.posterior_mean[2] = prior_mean + initial_bias / 2.0
        self.reward_var = float(reward_std ** 2)
        self.est_payoffs = self.posterior_mean.copy()
        self.K = np.zeros(arms, dtype=np.int64)

    def policy_action(self) -> int:
        sampled_means = self.rng.normal(self.posterior_mean, np.sqrt(self.posterior_var))
        return int(np.argmax(sampled_means))

    def policy_update(self, action: int, reward: float) -> None:
        old_mean = self.posterior_mean[action]
        old_var = self.posterior_var[action]
        posterior_precision = 1.0 / old_var + 1.0 / self.reward_var
        new_var = 1.0 / posterior_precision
        new_mean = new_var * (old_mean / old_var + reward / self.reward_var)
        self.posterior_mean[action] = new_mean
        self.posterior_var[action] = new_var
        self.K[action] += 1
        self.est_payoffs[action] = new_mean


print("ThompsonSampling class defined with reward_std=2.0 (pseudo-variance 4).")


In [ ]:
# =============================================================================
# CELL 8: REPAIRED ANYTIME EXP3 POLICY
# =============================================================================
# For N stages and raw payoff r, use gain g=(r-1)/(2N). At zero-indexed round t:
#   q_t(a) = w_t(a) / sum_b w_t(b)
#   pi_t(a) = (1-gamma_t) q_t(a) + gamma_t/N
#   gamma_t = d((t+1)+t0)^(-b)
#   eta_t = c((t+1)+t0)^(-2b)
#   g_hat_t(a) = 1{a=A_t} g_t / pi_t(a)
#   w_{t+1}(a) = w_t(a) exp(eta_t g_hat_t(a))
#
# Defaults are b=1/3, c=d=t0=1, so 0<gamma_t<1 from the first round.
# Updates use log weights and a harmless common recentering. No gain estimate or
# exponent is clipped. This is repaired anytime Exp3, not horizon-fixed Exp3.

class Exp3Policy:
    def __init__(self, arms: int, rng: np.random.Generator,
                 initial_bias: float = 0.0, b: float = 1/3,
                 c: float = 1.0, d: float = 1.0, t0: float = 1.0):
        if arms < 2:
            raise ValueError("Exp3 requires at least two actions")
        if not (0 < b <= 0.5) or c <= 0 or d <= 0 or t0 < 0:
            raise ValueError("Require 0<b<=1/2, c>0, d>0, and t0>=0")
        self.arms = arms
        self.rng = rng
        self.b, self.c, self.d, self.t0 = float(b), float(c), float(d), float(t0)
        initial_weights = np.full(arms, 1.0 / arms, dtype=np.float64)
        if arms >= 3 and initial_bias != 0.0:
            initial_weights[0] -= initial_bias
            initial_weights[2] += initial_bias
        if np.any(~np.isfinite(initial_weights)) or np.any(initial_weights <= 0):
            raise ValueError("Every initial Exp3 weight must be strictly positive")
        self.log_weights = np.log(initial_weights)
        self.timestep = 0
        self.last_probs = None
        self.current_gamma = None
        self.current_eta = None
        self.est_payoffs = np.zeros(arms, dtype=np.float64)
        self.K = np.zeros(arms, dtype=np.int64)

    @property
    def weights(self) -> np.ndarray:
        centered = self.log_weights - np.max(self.log_weights)
        weights = np.exp(centered)
        return weights / np.sum(weights)

    def _schedules(self):
        shifted_time = (self.timestep + 1) + self.t0
        gamma_t = self.d * shifted_time ** (-self.b)
        eta_t = self.c * shifted_time ** (-2 * self.b)
        if not (0.0 < gamma_t < 1.0):
            raise ValueError(f"gamma_t must lie in (0,1); observed {gamma_t}")
        return gamma_t, eta_t

    def _get_probabilities(self) -> np.ndarray:
        gamma_t, _ = self._schedules()
        q_t = self.weights
        return (1.0 - gamma_t) * q_t + gamma_t / self.arms

    def policy_action(self) -> int:
        self.current_gamma, self.current_eta = self._schedules()
        self.last_probs = self._get_probabilities()
        return int(self.rng.choice(self.arms, p=self.last_probs))

    def policy_update(self, action: int, reward: float) -> None:
        if self.last_probs is None or self.current_eta is None:
            raise RuntimeError("policy_action must precede policy_update")
        gain = (reward - 1.0) / (2.0 * self.arms)
        if not (0.0 <= gain <= 1.0):
            raise ValueError(f"Normalized gain outside [0,1]: {gain}")
        importance_estimate = gain / self.last_probs[action]
        self.log_weights[action] += self.current_eta * importance_estimate
        self.log_weights -= np.max(self.log_weights)
        self.K[action] += 1
        self.est_payoffs[action] += (reward - self.est_payoffs[action]) / self.K[action]
        self.timestep += 1
        self.last_probs = None
        self.current_gamma = None
        self.current_eta = None


print("Repaired anytime Exp3Policy class defined.")


In [ ]:
# =============================================================================
# CELL 9: POLICY PARAMETER CHECKS WHEN THE NOTEBOOK IS RUN
# =============================================================================
print("=" * 60)
print("POLICY PARAMETER CHECKS")
print("=" * 60)
test_rng = np.random.default_rng(12345)

eg_agent = EpsilonGreedyPolicy(arms=3, rng=test_rng, initial_bias=0.0)
print(f"Epsilon-greedy Q(0): {eg_agent.est_payoffs}; epsilon_0=1")

ts_agent = ThompsonSampling(
    arms=3, rng=test_rng, reward_std=2.0,
    prior_mean=0.0, prior_std=10.0, initial_bias=0.0,
)
print(f"Thompson means: {ts_agent.posterior_mean}; pseudo-variance={ts_agent.reward_var}")

exp3_agent = Exp3Policy(
    arms=3, rng=test_rng, initial_bias=0.0,
    b=1/3, c=1.0, d=1.0, t0=1.0,
)
gamma_0, eta_0 = exp3_agent._schedules()
print(f"Repaired anytime Exp3 weights: {exp3_agent.weights}")
print(f"First-round schedules: gamma_0={gamma_0:.8f}, eta_0={eta_0:.8f}")
assert np.allclose(exp3_agent.weights, np.full(3, 1/3))
assert 0.0 < gamma_0 < 1.0
print("Policy parameter checks complete.")


In [ ]:
# =============================================================================
# CELL 10: OUTCOME AND STABILITY DEFINITIONS
# =============================================================================
TERMINAL_FRACTION = 0.25
TERMINAL_SHARE_THRESHOLD = 0.75
FIRST_DOMINANCE_WINDOW = 500
FIRST_DOMINANCE_THRESHOLD = 0.90
DIAGNOSTIC_WINDOW_ROUNDS = 100_000
STABILITY_WINDOWS_REQUIRED = 3
STABILITY_SHARE_TOLERANCE = 0.05
STABLE_REPLICATE_FRACTION = 0.90


def classify_terminal_outcome(joint_action_counts, total_rounds=None,
                              threshold=TERMINAL_SHARE_THRESHOLD):
    """Classify terminal concentration from a complete 3x3 count matrix."""
    counts = np.asarray(joint_action_counts, dtype=np.int64)
    denominator = int(counts.sum()) if total_rounds is None else int(total_rounds)
    if counts.shape != (3, 3) or denominator <= 0:
        return 3
    for outcome_id in range(3):
        if counts[outcome_id, outcome_id] / denominator >= threshold:
            return outcome_id
    return 3


def normalized_entropy(probabilities):
    """Return entropy on [0,1], with zero-probability terms omitted."""
    p = np.asarray(probabilities, dtype=np.float64)
    p = p[p > 0]
    if len(p) <= 1:
        return 0.0
    return float(-np.sum(p * np.log(p)) / np.log(4.0))


print("Outcome and stability definitions loaded.")


In [ ]:
# =============================================================================
# CELL 11: REPLICATED, ADAPTIVE GRID-SETTING SIMULATION
# =============================================================================
def make_replicate_seed(base_seed, policy_type, replicate_index):
    """Create a policy-specific replicate seed shared across grid settings."""
    code = POLICY_SEED_CODES[policy_type]
    sequence = np.random.SeedSequence([int(base_seed), int(code), int(replicate_index)])
    return int(sequence.generate_state(1, dtype=np.uint64)[0])


def make_policy_pair(policy_type, n_actions, bias_p1, bias_p2, policy_kwargs, replicate_seed):
    player_sequences = np.random.SeedSequence(int(replicate_seed)).spawn(2)
    rng_p1 = np.random.default_rng(player_sequences[0])
    rng_p2 = np.random.default_rng(player_sequences[1])
    if policy_type == 'epsilon_greedy':
        return (
            EpsilonGreedyPolicy(n_actions, rng_p1, bias_p1),
            EpsilonGreedyPolicy(n_actions, rng_p2, bias_p2),
        )
    if policy_type == 'thompson':
        kwargs = {
            'reward_std': policy_kwargs.get('reward_std', 2.0),
            'prior_mean': policy_kwargs.get('prior_mean', 0.0),
            'prior_std': policy_kwargs.get('prior_std', 10.0),
        }
        return (
            ThompsonSampling(n_actions, rng_p1, initial_bias=bias_p1, **kwargs),
            ThompsonSampling(n_actions, rng_p2, initial_bias=bias_p2, **kwargs),
        )
    if policy_type == 'exp3':
        kwargs = {
            'b': policy_kwargs.get('b', 1/3), 'c': policy_kwargs.get('c', 1.0),
            'd': policy_kwargs.get('d', 1.0), 't0': policy_kwargs.get('t0', 1.0),
        }
        return (
            Exp3Policy(n_actions, rng_p1, initial_bias=bias_p1, **kwargs),
            Exp3Policy(n_actions, rng_p2, initial_bias=bias_p2, **kwargs),
        )
    raise ValueError(f"Unknown policy type: {policy_type}")


class ReplicateTracker:
    """Maintain one uninterrupted stochastic trajectory and compact summaries."""

    def __init__(self, policy_type, n_actions, bias_p1, bias_p2,
                 policy_kwargs, replicate_index, replicate_seed):
        self.policy_type = policy_type
        self.replicate_index = int(replicate_index)
        self.replicate_seed = int(replicate_seed)
        self.agent_p1, self.agent_p2 = make_policy_pair(
            policy_type, n_actions, bias_p1, bias_p2,
            policy_kwargs, replicate_seed,
        )
        self.n_actions = n_actions
        self.rounds_completed = 0
        self.block_joint_counts = []
        self.total_joint_counts = np.zeros((n_actions, n_actions), dtype=np.int64)
        self.dominance_window = deque(maxlen=FIRST_DOMINANCE_WINDOW)
        self.dominance_bi_count = 0
        self.currently_dominant = False
        self.first_dominance_time = -1
        self.last_dominance_exit = -1
        self.dominance_entries = 0
        self.dominance_exits = 0

    def run_to(self, target_rounds, rounds_per_block, game_local):
        if target_rounds < self.rounds_completed:
            raise ValueError("A trajectory cannot move backward in time")
        if target_rounds % rounds_per_block:
            raise ValueError("Every horizon must be divisible by rounds_per_block")
        while self.rounds_completed < target_rounds:
            block_counts = np.zeros((self.n_actions, self.n_actions), dtype=np.int32)
            for _ in range(rounds_per_block):
                action_p1 = self.agent_p1.policy_action()
                action_p2 = self.agent_p2.policy_action()
                reward_p1, reward_p2 = game_local.get_payoffs(action_p1, action_p2)
                self.agent_p1.policy_update(action_p1, float(reward_p1))
                self.agent_p2.policy_update(action_p2, float(reward_p2))
                block_counts[action_p1, action_p2] += 1
                self.total_joint_counts[action_p1, action_p2] += 1

                is_bi = int((action_p1, action_p2) == (0, 0))
                if len(self.dominance_window) == FIRST_DOMINANCE_WINDOW:
                    self.dominance_bi_count -= self.dominance_window[0]
                self.dominance_window.append(is_bi)
                self.dominance_bi_count += is_bi
                dominant_now = (
                    len(self.dominance_window) == FIRST_DOMINANCE_WINDOW
                    and self.dominance_bi_count / FIRST_DOMINANCE_WINDOW
                    >= FIRST_DOMINANCE_THRESHOLD
                )
                if dominant_now and not self.currently_dominant:
                    self.dominance_entries += 1
                    if self.first_dominance_time < 0:
                        self.first_dominance_time = (
                            self.rounds_completed + _ - FIRST_DOMINANCE_WINDOW + 1
                        )
                elif self.currently_dominant and not dominant_now:
                    self.dominance_exits += 1
                    self.last_dominance_exit = self.rounds_completed + _
                self.currently_dominant = dominant_now

            self.block_joint_counts.append(block_counts)
            self.rounds_completed += rounds_per_block

    def diagnostic_window_counts(self, rounds_per_block):
        blocks_per_window = DIAGNOSTIC_WINDOW_ROUNDS // rounds_per_block
        if blocks_per_window <= 0 or DIAGNOSTIC_WINDOW_ROUNDS % rounds_per_block:
            raise ValueError("Diagnostic window must be divisible by rounds_per_block")
        blocks = np.asarray(self.block_joint_counts, dtype=np.int64)
        complete = (len(blocks) // blocks_per_window) * blocks_per_window
        if complete == 0:
            return np.empty((0, self.n_actions, self.n_actions), dtype=np.int64)
        return blocks[:complete].reshape(
            -1, blocks_per_window, self.n_actions, self.n_actions
        ).sum(axis=1)

    def terminal_counts(self):
        n_tail_blocks = max(1, int(len(self.block_joint_counts) * TERMINAL_FRACTION))
        return np.asarray(self.block_joint_counts[-n_tail_blocks:], dtype=np.int64).sum(axis=0)

    def stable(self, rounds_per_block):
        windows = self.diagnostic_window_counts(rounds_per_block)
        if len(windows) < STABILITY_WINDOWS_REQUIRED:
            return False
        recent = windows[-STABILITY_WINDOWS_REQUIRED:]
        totals = recent.sum(axis=(1, 2), keepdims=True)
        shares = recent / totals
        labels = [classify_terminal_outcome(counts) for counts in recent]
        same_label = len(set(labels)) == 1
        share_drift = float(np.max(np.ptp(shares, axis=0)))
        return bool(same_label and share_drift <= STABILITY_SHARE_TOLERANCE)

    def final_policy_state(self):
        agents = (self.agent_p1, self.agent_p2)
        if self.policy_type == 'epsilon_greedy':
            return {
                'location': np.stack([agent.est_payoffs for agent in agents]),
                'scale': np.zeros((2, self.n_actions), dtype=np.float64),
                'counts': np.stack([agent.K for agent in agents]),
            }
        if self.policy_type == 'thompson':
            return {
                'location': np.stack([agent.posterior_mean for agent in agents]),
                'scale': np.stack([agent.posterior_var for agent in agents]),
                'counts': np.stack([agent.K for agent in agents]),
            }
        return {
            'location': np.stack([agent.weights for agent in agents]),
            'scale': np.stack([agent._get_probabilities() for agent in agents]),
            'counts': np.stack([agent.K for agent in agents]),
        }


def summarize_replicates(replicates, rounds_per_block):
    outcomes = np.array([
        classify_terminal_outcome(rep.terminal_counts()) for rep in replicates
    ], dtype=np.int8)
    outcome_counts = np.bincount(outcomes, minlength=4)
    outcome_probabilities = outcome_counts / len(replicates)
    stable_flags = np.array([
        rep.stable(rounds_per_block) for rep in replicates
    ], dtype=bool)
    return outcomes, outcome_counts, outcome_probabilities, stable_flags


def _simulate_grid_setting_impl(
    i, j, bias_p1, bias_p2, policy_type,
    payoff_matrix_p1, payoff_matrix_p2,
    rounds_per_block, horizon_schedule,
    min_replicates, max_replicates, replicate_batch_size,
    modal_probability_stop, policy_kwargs, base_seed,
):
    game_local = TwoPlayerGame(payoff_matrix_p1, payoff_matrix_p2)
    n_actions = game_local.n_actions
    replicates = []

    def add_replicates(number):
        start = len(replicates)
        for replicate_index in range(start, start + number):
            seed = make_replicate_seed(base_seed, policy_type, replicate_index)
            replicates.append(ReplicateTracker(
                policy_type, n_actions, bias_p1, bias_p2, policy_kwargs,
                replicate_index, seed,
            ))

    add_replicates(min_replicates)
    horizon_index = 0
    stop_reason = 'maximum horizon reached'

    while True:
        target_horizon = int(horizon_schedule[horizon_index])
        for replicate in replicates:
            replicate.run_to(target_horizon, rounds_per_block, game_local)

        outcomes, outcome_counts, outcome_probabilities, stable_flags = summarize_replicates(
            replicates, rounds_per_block
        )
        stable_fraction = float(np.mean(stable_flags))
        needs_more_replicates = (
            np.max(outcome_probabilities) < modal_probability_stop
            and len(replicates) < max_replicates
        )

        if stable_fraction >= STABLE_REPLICATE_FRACTION:
            if needs_more_replicates:
                add_replicates(min(replicate_batch_size, max_replicates - len(replicates)))
                continue
            stop_reason = 'window-level behavior stabilized'
            break

        if horizon_index + 1 < len(horizon_schedule):
            horizon_index += 1
            continue

        if needs_more_replicates:
            add_replicates(min(replicate_batch_size, max_replicates - len(replicates)))
            continue
        break

    outcomes, outcome_counts, outcome_probabilities, stable_flags = summarize_replicates(
        replicates, rounds_per_block
    )
    final_horizon = int(horizon_schedule[horizon_index])
    terminal_counts = np.stack([rep.terminal_counts() for rep in replicates])
    total_counts = np.stack([rep.total_joint_counts for rep in replicates])
    diagnostic_counts = np.stack([
        rep.diagnostic_window_counts(rounds_per_block) for rep in replicates
    ])
    final_states = [rep.final_policy_state() for rep in replicates]
    first_times = np.array([rep.first_dominance_time for rep in replicates], dtype=np.int64)

    return {
        'metadata': {
            'i': int(i), 'j': int(j), 'policy': policy_type,
            'bias_p1': float(bias_p1), 'bias_p2': float(bias_p2),
            'n_replicates': len(replicates), 'final_horizon': final_horizon,
            'stop_reason': stop_reason,
        },
        'replicate_indices': np.array([rep.replicate_index for rep in replicates], dtype=np.int16),
        'replicate_seeds': np.array([rep.replicate_seed for rep in replicates], dtype=np.uint64),
        'outcome_ids': outcomes,
        'outcome_counts': outcome_counts.astype(np.int16),
        'outcome_probabilities': outcome_probabilities.astype(np.float64),
        'stable_flags': stable_flags,
        'terminal_joint_counts': terminal_counts,
        'total_joint_counts': total_counts,
        'diagnostic_joint_counts': diagnostic_counts,
        'first_dominance_times': first_times,
        'last_dominance_exits': np.array([rep.last_dominance_exit for rep in replicates], dtype=np.int64),
        'dominance_entries': np.array([rep.dominance_entries for rep in replicates], dtype=np.int32),
        'dominance_exits': np.array([rep.dominance_exits for rep in replicates], dtype=np.int32),
        'final_state_location': np.stack([state['location'] for state in final_states]),
        'final_state_scale': np.stack([state['scale'] for state in final_states]),
        'final_action_counts': np.stack([state['counts'] for state in final_states]),
    }


simulate_grid_setting = ray.remote(_simulate_grid_setting_impl)  # type: ignore
print("Replicated adaptive grid-setting simulation defined.")


In [ ]:
# =============================================================================
# CELL 12: RESUMABLE MAP CONSTRUCTION AND ATOMIC STORAGE
# =============================================================================
def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent,
                                     prefix=path.name, suffix='.tmp', delete=False) as stream:
        json.dump(payload, stream, indent=2, sort_keys=True)
        stream.write('\n')
        temporary = Path(stream.name)
    os.replace(temporary, path)


def atomic_write_npz(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    serializable = {key: value for key, value in payload.items() if key != 'metadata'}
    serializable['metadata_json'] = np.array(json.dumps(payload['metadata'], sort_keys=True))
    with tempfile.NamedTemporaryFile('wb', dir=path.parent, prefix=path.name,
                                     suffix='.tmp', delete=False) as stream:
        np.savez_compressed(stream, **serializable)
        temporary = Path(stream.name)
    os.replace(temporary, path)


def load_grid_result(path):
    with np.load(path, allow_pickle=False) as archive:
        result = {key: archive[key] for key in archive.files if key != 'metadata_json'}
        result['metadata'] = json.loads(str(archive['metadata_json'].item()))
    return result


def notebook_source_hash(path):
    notebook = json.loads(Path(path).read_text(encoding='utf-8'))
    source_payload = [
        {'cell_type': cell.get('cell_type'), 'source': cell.get('source', [])}
        for cell in notebook.get('cells', [])
    ]
    encoded = json.dumps(source_payload, ensure_ascii=False, sort_keys=True).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


def aggregate_grid_result(aggregate, result):
    metadata = result['metadata']
    i, j = metadata['i'], metadata['j']
    probabilities = result['outcome_probabilities']
    aggregate['outcome_probability_maps'][:, i, j] = probabilities
    aggregate['modal_outcome_map'][i, j] = int(np.argmax(probabilities))
    aggregate['outcome_entropy_map'][i, j] = normalized_entropy(probabilities)
    aggregate['replicate_count_map'][i, j] = metadata['n_replicates']
    aggregate['final_horizon_map'][i, j] = metadata['final_horizon']
    aggregate['stable_fraction_map'][i, j] = float(np.mean(result['stable_flags']))
    times = result['first_dominance_times']
    finite_times = times[times >= 0]
    aggregate['mean_first_dominance_map'][i, j] = (
        float(np.mean(finite_times)) if len(finite_times) else np.nan
    )


def construct_stochastic_attraction_map(
    policy_type, payoff_matrix_p1, payoff_matrix_p2, run_dir,
    grid_resolution, rounds_per_block, param_range, horizon_schedule,
    min_replicates, max_replicates, replicate_batch_size,
    modal_probability_stop, policy_kwargs=None, base_seed=BASE_SEED,
):
    """Estimate outcome probabilities at every point of an initial-bias slice."""
    policy_kwargs = {} if policy_kwargs is None else dict(policy_kwargs)
    run_dir = Path(run_dir)
    cell_dir = run_dir / 'cells' / policy_type
    cell_dir.mkdir(parents=True, exist_ok=True)
    param_values = np.linspace(param_range[0], param_range[1], grid_resolution)
    shape = (grid_resolution, grid_resolution)
    aggregate = {
        'outcome_probability_maps': np.full((4,) + shape, np.nan, dtype=np.float64),
        'modal_outcome_map': np.full(shape, -1, dtype=np.int8),
        'outcome_entropy_map': np.full(shape, np.nan, dtype=np.float64),
        'replicate_count_map': np.zeros(shape, dtype=np.int16),
        'final_horizon_map': np.zeros(shape, dtype=np.int64),
        'stable_fraction_map': np.full(shape, np.nan, dtype=np.float64),
        'mean_first_dominance_map': np.full(shape, np.nan, dtype=np.float64),
    }

    futures = []
    completed = 0
    total_settings = grid_resolution ** 2
    for i, bias_p1 in enumerate(param_values):
        for j, bias_p2 in enumerate(param_values):
            cell_path = cell_dir / f'cell_{i:02d}_{j:02d}.npz'
            if cell_path.exists():
                aggregate_grid_result(aggregate, load_grid_result(cell_path))
                completed += 1
                continue
            future = simulate_grid_setting.remote(
                i, j, bias_p1, bias_p2, policy_type,
                payoff_matrix_p1, payoff_matrix_p2,
                rounds_per_block, tuple(horizon_schedule),
                min_replicates, max_replicates, replicate_batch_size,
                modal_probability_stop, policy_kwargs, base_seed,
            )
            futures.append(future)

    print(f"{policy_type}: {completed}/{total_settings} cells restored; {len(futures)} scheduled")
    started = time()
    while futures:
        done, futures = ray.wait(futures, num_returns=min(len(futures), 25), timeout=5.0)
        for result in ray.get(done):
            metadata = result['metadata']
            cell_path = cell_dir / f"cell_{metadata['i']:02d}_{metadata['j']:02d}.npz"
            atomic_write_npz(cell_path, result)
            aggregate_grid_result(aggregate, result)
            completed += 1
        elapsed = time() - started
        rate = completed / elapsed if elapsed > 0 else 0.0
        remaining = (total_settings - completed) / rate if rate > 0 else 0.0
        print(
            f"\rProgress: {completed}/{total_settings} ({100*completed/total_settings:.1f}%) "
            f"| elapsed {elapsed:.1f}s | ETA {remaining:.1f}s",
            end='',
        )
    print()

    aggregate_payload = {
        'metadata': {
            'policy': policy_type,
            'param_range': list(param_range),
            'grid_resolution': grid_resolution,
        },
        'param_values': param_values,
        **aggregate,
    }
    atomic_write_npz(run_dir / 'aggregate' / f'{policy_type}.npz', aggregate_payload)
    return aggregate, param_values


print("Resumable stochastic-attraction map construction defined.")


In [ ]:
# =============================================================================
# CELL 13: STOCHASTIC ATTRACTION VISUALIZATION
# =============================================================================
TERMINAL_OUTCOME_NAMES = {
    0: '(0,0) Backward induction',
    1: '(1,1) Terminal outcome',
    2: '(2,2) Terminal outcome',
    3: 'Mixed/other terminal outcome',
}
TERMINAL_OUTCOME_COLORS = ['#3498db', '#f39c12', '#2ecc71', '#95a5a6']


def plot_stage0_attraction_probability(probability_map, param_values, policy_name,
                                       param_name, save_path=None):
    """Plot the estimated probability of terminal concentration on (0,0)."""
    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(
        probability_map.T, origin='lower', cmap='viridis', vmin=0.0, vmax=1.0,
        extent=(param_values[0], param_values[-1], param_values[0], param_values[-1]),
        aspect='auto', interpolation='nearest',
    )
    plt.colorbar(im, ax=ax, label='Estimated probability of terminal (0,0) concentration')
    ax.axhline(0, color='white', linestyle='--', linewidth=1.2, alpha=0.7)
    ax.axvline(0, color='white', linestyle='--', linewidth=1.2, alpha=0.7)
    ax.set_xlabel(f'Player 1 {param_name}')
    ax.set_ylabel(f'Player 2 {param_name}')
    ax.set_title(f'Stochastic Attraction to (0,0): {policy_name}')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    return fig


def plot_modal_outcome_map(modal_map, entropy_map, param_values, policy_name,
                           param_name, save_path=None):
    """Plot the most frequent terminal outcome and mark uncertain cells."""
    fig, ax = plt.subplots(figsize=(10, 9))
    cmap = ListedColormap(TERMINAL_OUTCOME_COLORS)
    ax.imshow(
        modal_map.T, origin='lower', cmap=cmap, vmin=0, vmax=3,
        extent=(param_values[0], param_values[-1], param_values[0], param_values[-1]),
        aspect='auto', interpolation='nearest',
    )
    uncertain = np.argwhere(entropy_map > 0.35)
    if len(uncertain):
        x = param_values[uncertain[:, 0]]
        y = param_values[uncertain[:, 1]]
        ax.scatter(x, y, marker='.', color='black', s=8, alpha=0.55, label='Higher outcome uncertainty')
    legend = [Patch(facecolor=TERMINAL_OUTCOME_COLORS[i], label=TERMINAL_OUTCOME_NAMES[i]) for i in range(4)]
    ax.legend(handles=legend, loc='upper right', fontsize=9)
    ax.set_xlabel(f'Player 1 {param_name}')
    ax.set_ylabel(f'Player 2 {param_name}')
    ax.set_title(f'Modal Terminal Outcome: {policy_name}')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    return fig


print("Stochastic attraction visualizations defined.")


In [ ]:
# =============================================================================
# CELL 14: HORIZON AND FIRST-DOMINANCE DIAGNOSTICS
# =============================================================================
def plot_scalar_grid(value_map, param_values, policy_name, param_name,
                     title, colorbar_label, save_path=None, cmap='viridis_r'):
    fig, ax = plt.subplots(figsize=(10, 9))
    masked = np.ma.masked_invalid(value_map)
    im = ax.imshow(
        masked.T, origin='lower', cmap=cmap,
        extent=(param_values[0], param_values[-1], param_values[0], param_values[-1]),
        aspect='auto', interpolation='nearest',
    )
    plt.colorbar(im, ax=ax, label=colorbar_label)
    ax.set_xlabel(f'Player 1 {param_name}')
    ax.set_ylabel(f'Player 2 {param_name}')
    ax.set_title(f'{title}: {policy_name}')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    return fig


print("Horizon and first-dominance diagnostics defined.")


In [ ]:
# =============================================================================
# CELL 15: ACTION-PROBABILITY HELPERS
# =============================================================================
def get_epsilon_greedy_probs(agent, timestep):
    epsilon = 1.0 / (timestep + 1)
    max_q = np.max(agent.est_payoffs)
    best_actions = np.flatnonzero(np.isclose(agent.est_payoffs, max_q, rtol=0.0, atol=1e-10))
    probs = np.full(agent.arms, epsilon / agent.arms)
    probs[best_actions] += (1.0 - epsilon) / len(best_actions)
    return probs


def get_thompson_probs(agent, rng: np.random.Generator, n_samples=1000):
    """Estimate current TS selection probabilities with a supplied seeded RNG."""
    samples = rng.normal(
        agent.posterior_mean[:, np.newaxis],
        np.sqrt(agent.posterior_var)[:, np.newaxis],
        size=(agent.arms, n_samples),
    )
    best_actions = np.argmax(samples, axis=0)
    return np.bincount(best_actions, minlength=agent.arms) / n_samples


def get_exp3_probs(agent):
    return agent._get_probabilities()


print("Action-probability helpers defined.")


In [ ]:
# =============================================================================
# CELL 15: SIMPLEX COORDINATE TRANSFORMATION
# =============================================================================
# Convert probability vectors to 2D Cartesian coordinates for plotting

def barycentric_to_cartesian(probs):
    """
    Convert 3D probability vector to 2D Cartesian coordinates.
    
    For probability vector [p0, p1, p2], maps to equilateral triangle:
    - Vertex 0 (Take=0): bottom-left  (0, 0)
    - Vertex 1 (Take=1): top          (0.5, sqrt(3)/2)
    - Vertex 2 (Take=2): bottom-right (1, 0)
    """
    x = probs[2] + 0.5 * probs[1]
    y = (np.sqrt(3) / 2) * probs[1]
    return x, y


def plot_simplex_frame(ax, labels=None):
    """
    Draw the simplex triangle frame.
    """
    if labels is None:
        labels = ['Take=0', 'Take=1', 'Take=2']
    
    # Draw triangle edges
    triangle = np.array([[0, 0], [0.5, np.sqrt(3)/2], [1, 0], [0, 0]])
    ax.plot(triangle[:, 0], triangle[:, 1], 'k-', linewidth=2)
    
    # Mark vertices
    vertices = triangle[:-1]
    ax.plot(vertices[:, 0], vertices[:, 1], 'ko', markersize=10, zorder=5)
    
    # Add labels with offset
    offset = 0.08
    positions = [
        (vertices[0, 0] - offset, vertices[0, 1] - offset),
        (vertices[1, 0], vertices[1, 1] + offset),
        (vertices[2, 0] + offset, vertices[2, 1] - offset)
    ]
    
    for (x, y), label in zip(positions, labels):
        ax.text(x, y, label, fontsize=12, fontweight='bold',
                ha='center', va='center')
    
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_xlim(-0.15, 1.15)
    ax.set_ylim(-0.15, 1.0)


print("Simplex transformation functions defined.")

In [ ]:
# =============================================================================
# CELL 17: SINGLE CONTINUOUS DIAGNOSTIC TRAJECTORY
# =============================================================================
def simulate_trajectory(
    policy_type, bias_p1, bias_p2, payoff_matrix_p1, payoff_matrix_p2,
    n_blocks, rounds_per_block, checkpoint_interval=100,
    policy_kwargs=None, run_seed=None,
):
    """Run one trajectory and retain exact joint counts between checkpoints."""
    if run_seed is None:
        raise ValueError("A deterministic run_seed is required")
    policy_kwargs = {} if policy_kwargs is None else dict(policy_kwargs)
    seed_children = np.random.SeedSequence(int(run_seed)).spawn(4)
    rng_p1, rng_p2, diagnostic_rng_p1, diagnostic_rng_p2 = [
        np.random.default_rng(seed) for seed in seed_children
    ]
    game_local = TwoPlayerGame(payoff_matrix_p1, payoff_matrix_p2)
    n_actions = game_local.n_actions
    agent_p1, agent_p2 = make_policy_pair(
        policy_type, n_actions, bias_p1, bias_p2, policy_kwargs, int(run_seed)
    )

    if policy_type == 'epsilon_greedy':
        probability_fn = lambda agent, diag_rng: get_epsilon_greedy_probs(agent, agent.timestep)
    elif policy_type == 'thompson':
        probability_fn = get_thompson_probs
    elif policy_type == 'exp3':
        probability_fn = lambda agent, diag_rng: get_exp3_probs(agent)
    else:
        raise ValueError(f"Unknown policy type: {policy_type}")

    timesteps, probs_p1, probs_p2 = [], [], []
    states_p1, states_p2, interval_joint_counts = [], [], []
    interval_counts = np.zeros((n_actions, n_actions), dtype=np.int64)
    total_rounds = n_blocks * rounds_per_block
    for step in range(total_rounds):
        action_p1 = agent_p1.policy_action()
        action_p2 = agent_p2.policy_action()
        reward_p1, reward_p2 = game_local.get_payoffs(action_p1, action_p2)
        agent_p1.policy_update(action_p1, float(reward_p1))
        agent_p2.policy_update(action_p2, float(reward_p2))
        interval_counts[action_p1, action_p2] += 1

        if (step + 1) % checkpoint_interval == 0:
            timesteps.append(step + 1)
            interval_joint_counts.append(interval_counts.copy())
            interval_counts.fill(0)
            probs_p1.append(probability_fn(agent_p1, diagnostic_rng_p1))
            probs_p2.append(probability_fn(agent_p2, diagnostic_rng_p2))
            if policy_type == 'thompson':
                states_p1.append({'mean': agent_p1.posterior_mean.copy(), 'var': agent_p1.posterior_var.copy()})
                states_p2.append({'mean': agent_p2.posterior_mean.copy(), 'var': agent_p2.posterior_var.copy()})
            else:
                states_p1.append(agent_p1.est_payoffs.copy() if policy_type == 'epsilon_greedy' else agent_p1.weights.copy())
                states_p2.append(agent_p2.est_payoffs.copy() if policy_type == 'epsilon_greedy' else agent_p2.weights.copy())

    return {
        'timesteps': np.asarray(timesteps),
        'probs_p1': np.asarray(probs_p1), 'probs_p2': np.asarray(probs_p2),
        'interval_joint_counts': np.asarray(interval_joint_counts),
        'states_p1': states_p1, 'states_p2': states_p2,
        'policy_type': policy_type, 'run_seed': int(run_seed),
        'checkpoint_interval': int(checkpoint_interval),
    }


print("simulate_trajectory() defined with exact checkpoint-window counts.")


In [ ]:
# =============================================================================
# CELL 18: TRAJECTORY ENSEMBLE
# =============================================================================
def simulate_trajectory_ensemble(
    policy_type, bias_p1, bias_p2, payoff_matrix_p1, payoff_matrix_p2,
    n_blocks, rounds_per_block, n_runs=20, checkpoint_interval=100,
    policy_kwargs=None, base_seed=BASE_SEED,
):
    """Run multiple independently seeded continuous trajectories."""
    child_sequences = np.random.SeedSequence(base_seed).spawn(n_runs)
    run_seeds = [int(seq.generate_state(1, dtype=np.uint64)[0]) for seq in child_sequences]
    ensemble = []
    for run_index, run_seed in enumerate(run_seeds):
        if (run_index + 1) % 5 == 0:
            print(f"Completed {run_index + 1}/{n_runs} trajectories")
        ensemble.append(simulate_trajectory(
            policy_type, bias_p1, bias_p2, payoff_matrix_p1, payoff_matrix_p2,
            n_blocks, rounds_per_block, checkpoint_interval, policy_kwargs,
            run_seed=run_seed,
        ))
    return ensemble


print("simulate_trajectory_ensemble() defined.")


In [ ]:
# =============================================================================
# CELL 18: TRAJECTORY PLOTTING - SINGLE
# =============================================================================
# Plot individual player trajectory in probability simplex

def plot_aps_trajectory(trajectory_data, player='P1', ax=None, 
                       color='blue', alpha=0.7, show_points=True):
    """
    Plot Action Probability Simplex trajectory for one player.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    
    probs = trajectory_data[f'probs_{player.lower()}']
    coords = np.array([barycentric_to_cartesian(p) for p in probs])
    
    plot_simplex_frame(ax, labels=['Take=0\n(BI)', 'Take=1', 'Take=2\n(Coop)'])
    
    ax.plot(coords[:, 0], coords[:, 1], '-', color=color, alpha=alpha, 
            linewidth=2, label=f'{player} trajectory')
    
    ax.plot(coords[0, 0], coords[0, 1], 'o', color='green', 
            markersize=12, label='Start', zorder=10)
    ax.plot(coords[-1, 0], coords[-1, 1], '*', color='red', 
            markersize=18, label='End', zorder=10)
    
    if show_points and len(coords) > 2:
        ax.scatter(coords[1:-1, 0], coords[1:-1, 1], c=color, 
                  alpha=0.3, s=30, zorder=5)
    
    ax.legend(loc='upper right', fontsize=10)
    ax.set_title(f'Action Probability Simplex - {player}', 
                fontsize=14, fontweight='bold')
    
    return ax


print("plot_aps_trajectory() function defined.")

In [ ]:
# =============================================================================
# CELL 19: TRAJECTORY PLOTTING - ENSEMBLE
# =============================================================================
# Plot multiple trajectories as overlays (spaghetti plot)

def plot_trajectory_ensemble(ensemble, player='P1', ax=None,
                             color='blue', alpha=0.15, show_mean=True):
    """
    Plot ensemble of trajectories as transparent overlays.
    
    Parameters
    ----------
    ensemble : list of dict
        Output from simulate_trajectory_ensemble()
    player : str
        'P1' or 'P2'
    ax : matplotlib.axes.Axes, optional
        Axes to plot on
    color : str
        Line color
    alpha : float
        Individual trajectory transparency
    show_mean : bool
        Whether to show mean trajectory
    
    Returns
    -------
    matplotlib.axes.Axes
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    
    plot_simplex_frame(ax, labels=['Take=0\n(BI)', 'Take=1', 'Take=2\n(Coop)'])
    
    # Plot individual trajectories
    all_coords = []
    for traj in ensemble:
        probs = traj[f'probs_{player.lower()}']
        coords = np.array([barycentric_to_cartesian(p) for p in probs])
        all_coords.append(coords)
        ax.plot(coords[:, 0], coords[:, 1], '-', color=color, 
                alpha=alpha, linewidth=1)
    
    # Plot mean trajectory if requested
    if show_mean and len(all_coords) > 0:
        # Ensure all trajectories have same length for averaging
        min_len = min(len(c) for c in all_coords)
        coords_array = np.array([c[:min_len] for c in all_coords])
        mean_coords = np.mean(coords_array, axis=0)
        
        ax.plot(mean_coords[:, 0], mean_coords[:, 1], '-', color='black', 
                alpha=1.0, linewidth=3, label='Mean trajectory')
    
    # Mark common start and end regions
    start_x = np.mean([c[0, 0] for c in all_coords])
    start_y = np.mean([c[0, 1] for c in all_coords])
    end_x = np.mean([c[-1, 0] for c in all_coords])
    end_y = np.mean([c[-1, 1] for c in all_coords])
    
    ax.plot(start_x, start_y, 'o', color='green', markersize=15, 
            label='Start (mean)', zorder=10, markeredgecolor='white', markeredgewidth=2)
    ax.plot(end_x, end_y, '*', color='red', markersize=20, 
            label='End (mean)', zorder=10, markeredgecolor='white', markeredgewidth=1)
    
    ax.legend(loc='upper right', fontsize=10)
    ax.set_title(f'Trajectory Ensemble ({len(ensemble)} runs) - {player}', 
                fontsize=14, fontweight='bold')
    
    return ax


print("plot_trajectory_ensemble() function defined.")

In [ ]:
# =============================================================================
# CELL 21: JOINT-ACTION FREQUENCIES OVER TIME
# =============================================================================
def plot_joint_action_frequencies(trajectory_data, ax=None):
    """Plot exact checkpoint-window shares, retaining off-diagonal outcomes."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 6))
    counts = np.asarray(trajectory_data['interval_joint_counts'], dtype=np.float64)
    totals = counts.sum(axis=(1, 2))
    timesteps = trajectory_data['timesteps']
    shares = {
        '(0,0)': counts[:, 0, 0] / totals,
        '(1,1)': counts[:, 1, 1] / totals,
        '(2,2)': counts[:, 2, 2] / totals,
    }
    diagonal = shares['(0,0)'] + shares['(1,1)'] + shares['(2,2)']
    shares['Off diagonal'] = 1.0 - diagonal
    colors = {'(0,0)': '#3498db', '(1,1)': '#f39c12', '(2,2)': '#2ecc71', 'Off diagonal': '#7f8c8d'}
    for label, values in shares.items():
        ax.plot(timesteps, values, label=label, color=colors[label], linewidth=1.8)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Round')
    ax.set_ylabel('Share within checkpoint window')
    ax.set_title('Joint-Action Frequencies')
    ax.legend()
    return ax


print("Exact joint-action-frequency plotting defined.")


In [ ]:
# =============================================================================
# CELL 22: COMBINED TRAJECTORY VISUALIZATION
# =============================================================================
def plot_complete_trajectory(trajectory_data, policy_name, save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    plot_aps_trajectory(trajectory_data, player='P1', ax=axes[0], color='dodgerblue')
    plot_aps_trajectory(trajectory_data, player='P2', ax=axes[1], color='coral')
    plot_joint_action_frequencies(trajectory_data, ax=axes[2])
    fig.suptitle(f'Learning Dynamics: {policy_name}\nQuasi-Centipede Game', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    return fig


def plot_complete_ensemble(ensemble, policy_name, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    plot_trajectory_ensemble(ensemble, player='P1', ax=axes[0], color='dodgerblue')
    plot_trajectory_ensemble(ensemble, player='P2', ax=axes[1], color='coral')
    fig.suptitle(f'Trajectory Ensemble: {policy_name}\nQuasi-Centipede Game ({len(ensemble)} runs)', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    return fig


print("Combined visualization functions defined.")


In [ ]:
# =============================================================================
# CELL 23: EXPERIMENT AND RUN-DIRECTORY CONFIGURATION
# =============================================================================
RUN_MODE = 'full'
RUN_CONFIGS = {
    'ultra_fast': {'grid_resolution': 3, 'rounds_per_block': 5_000, 'horizons': (100_000, 200_000), 'min_replicates': 2, 'max_replicates': 2},
    'debug': {'grid_resolution': 5, 'rounds_per_block': 5_000, 'horizons': (200_000, 500_000), 'min_replicates': 4, 'max_replicates': 8},
    'standard': {'grid_resolution': 20, 'rounds_per_block': 5_000, 'horizons': (500_000, 1_000_000, 2_000_000), 'min_replicates': 8, 'max_replicates': 16},
    'full': {
        'grid_resolution': 50,
        'rounds_per_block': 5_000,
        'horizons': (500_000, 1_000_000, 2_000_000, 4_000_000, 8_000_000, 10_000_000),
        'min_replicates': 16,
        'max_replicates': 32,
    },
}
config = RUN_CONFIGS[RUN_MODE]
GRID_RESOLUTION = config['grid_resolution']
ROUNDS_PER_BLOCK = config['rounds_per_block']
HORIZON_SCHEDULE = config['horizons']
MIN_REPLICATES = config['min_replicates']
MAX_REPLICATES = config['max_replicates']
REPLICATE_BATCH_SIZE = 8
MODAL_PROBABILITY_STOP = 0.85

PARAM_RANGES = {
    'epsilon_greedy': (-2.0, 2.0),
    'thompson': (-0.5, 0.5),
    'exp3': (-0.3, 0.3),
}
POLICY_KWARGS = {
    'epsilon_greedy': {},
    'thompson': {'reward_std': 2.0, 'prior_mean': 0.0, 'prior_std': 10.0},
    'exp3': {'b': 1/3, 'c': 1.0, 'd': 1.0, 't0': 1.0},
}

NOTEBOOK_PATH = Path('Centipede_Basins_Attraction_v2.ipynb').resolve()
SOURCE_HASH = notebook_source_hash(NOTEBOOK_PATH)
RESUME_RUN_ID = os.environ.get('CENTIPEDE_BASIN_RUN_ID')
RUN_STARTED_AT = datetime.now(timezone.utc)
RUN_ID = RESUME_RUN_ID or f"{RUN_STARTED_AT.strftime('%Y%m%dT%H%M%SZ')}_{SOURCE_HASH[:12]}"
RUN_DIR = Path('basin_runs') / RUN_ID
if RESUME_RUN_ID:
    if not (RUN_DIR / 'manifest.json').exists():
        raise FileNotFoundError(f"Cannot resume missing run: {RUN_DIR}")
else:
    for subdirectory in ('cells', 'aggregate', 'figures', 'logs'):
        (RUN_DIR / subdirectory).mkdir(parents=True, exist_ok=False)
    manifest = {
        'schema_version': 'centipede.stochastic-attraction.v3',
        'status': 'configured',
        'run_id': RUN_ID,
        'started_at_utc': RUN_STARTED_AT.isoformat(),
        'source_hash': SOURCE_HASH,
        'notebook': NOTEBOOK_PATH.name,
        'run_mode': RUN_MODE,
        'grid_resolution': GRID_RESOLUTION,
        'rounds_per_block': ROUNDS_PER_BLOCK,
        'horizon_schedule': list(HORIZON_SCHEDULE),
        'min_replicates': MIN_REPLICATES,
        'max_replicates': MAX_REPLICATES,
        'replicate_batch_size': REPLICATE_BATCH_SIZE,
        'modal_probability_stop': MODAL_PROBABILITY_STOP,
        'stable_replicate_fraction': STABLE_REPLICATE_FRACTION,
        'diagnostic_window_rounds': DIAGNOSTIC_WINDOW_ROUNDS,
        'stability_windows_required': STABILITY_WINDOWS_REQUIRED,
        'stability_share_tolerance': STABILITY_SHARE_TOLERANCE,
        'terminal_fraction': TERMINAL_FRACTION,
        'terminal_share_threshold': TERMINAL_SHARE_THRESHOLD,
        'first_dominance_window': FIRST_DOMINANCE_WINDOW,
        'first_dominance_threshold': FIRST_DOMINANCE_THRESHOLD,
        'base_seed': BASE_SEED,
        'seed_method': SEED_METHOD,
        'parameter_ranges': {key: list(value) for key, value in PARAM_RANGES.items()},
        'policy_parameters': POLICY_KWARGS,
    }
    atomic_write_json(RUN_DIR / 'manifest.json', manifest)

print(f"Run directory: {RUN_DIR.resolve()}")
print(f"Grid: {GRID_RESOLUTION} x {GRID_RESOLUTION}; initial replicates: {MIN_REPLICATES}")
print(f"Adaptive horizons: {HORIZON_SCHEDULE}")


In [ ]:
# =============================================================================
# RUN EPSILON-GREEDY STOCHASTIC ATTRACTION MAP
# =============================================================================
print("CONSTRUCTING EPSILON-GREEDY STOCHASTIC ATTRACTION MAP")
aggregate_epsilon_greedy, params_epsilon_greedy = construct_stochastic_attraction_map(
    policy_type='epsilon_greedy',
    payoff_matrix_p1=game.payoff_matrix_p1,
    payoff_matrix_p2=game.payoff_matrix_p2,
    run_dir=RUN_DIR,
    grid_resolution=GRID_RESOLUTION,
    rounds_per_block=ROUNDS_PER_BLOCK,
    param_range=PARAM_RANGES['epsilon_greedy'],
    horizon_schedule=HORIZON_SCHEDULE,
    min_replicates=MIN_REPLICATES,
    max_replicates=MAX_REPLICATES,
    replicate_batch_size=REPLICATE_BATCH_SIZE,
    modal_probability_stop=MODAL_PROBABILITY_STOP,
    policy_kwargs=POLICY_KWARGS['epsilon_greedy'],
    base_seed=BASE_SEED,
)
plot_stage0_attraction_probability(
    aggregate_epsilon_greedy['outcome_probability_maps'][0], params_epsilon_greedy,
    'Epsilon-Greedy', 'early-to-late Q-bias (beta)',
    save_path=RUN_DIR / 'figures' / 'epsilon_greedy_stage0_attraction.png',
)
plot_modal_outcome_map(
    aggregate_epsilon_greedy['modal_outcome_map'], aggregate_epsilon_greedy['outcome_entropy_map'],
    params_epsilon_greedy, 'Epsilon-Greedy', 'early-to-late Q-bias (beta)',
    save_path=RUN_DIR / 'figures' / 'epsilon_greedy_modal_outcome.png',
)
plot_scalar_grid(
    aggregate_epsilon_greedy['final_horizon_map'], params_epsilon_greedy,
    'Epsilon-Greedy', 'early-to-late Q-bias (beta)', 'Adaptive Final Horizon', 'Rounds',
    save_path=RUN_DIR / 'figures' / 'epsilon_greedy_final_horizon.png', cmap='magma',
)


In [ ]:
# =============================================================================
# RUN THOMPSON SAMPLING STOCHASTIC ATTRACTION MAP
# =============================================================================
print("CONSTRUCTING THOMPSON SAMPLING STOCHASTIC ATTRACTION MAP")
aggregate_thompson, params_thompson = construct_stochastic_attraction_map(
    policy_type='thompson',
    payoff_matrix_p1=game.payoff_matrix_p1,
    payoff_matrix_p2=game.payoff_matrix_p2,
    run_dir=RUN_DIR,
    grid_resolution=GRID_RESOLUTION,
    rounds_per_block=ROUNDS_PER_BLOCK,
    param_range=PARAM_RANGES['thompson'],
    horizon_schedule=HORIZON_SCHEDULE,
    min_replicates=MIN_REPLICATES,
    max_replicates=MAX_REPLICATES,
    replicate_batch_size=REPLICATE_BATCH_SIZE,
    modal_probability_stop=MODAL_PROBABILITY_STOP,
    policy_kwargs=POLICY_KWARGS['thompson'],
    base_seed=BASE_SEED,
)
plot_stage0_attraction_probability(
    aggregate_thompson['outcome_probability_maps'][0], params_thompson,
    'Thompson Sampling', 'early-to-late prior-mean bias (mu)',
    save_path=RUN_DIR / 'figures' / 'thompson_stage0_attraction.png',
)
plot_modal_outcome_map(
    aggregate_thompson['modal_outcome_map'], aggregate_thompson['outcome_entropy_map'],
    params_thompson, 'Thompson Sampling', 'early-to-late prior-mean bias (mu)',
    save_path=RUN_DIR / 'figures' / 'thompson_modal_outcome.png',
)
plot_scalar_grid(
    aggregate_thompson['final_horizon_map'], params_thompson,
    'Thompson Sampling', 'early-to-late prior-mean bias (mu)', 'Adaptive Final Horizon', 'Rounds',
    save_path=RUN_DIR / 'figures' / 'thompson_final_horizon.png', cmap='magma',
)


In [ ]:
# =============================================================================
# RUN REPAIRED ANYTIME EXP3 STOCHASTIC ATTRACTION MAP
# =============================================================================
print("CONSTRUCTING REPAIRED ANYTIME EXP3 STOCHASTIC ATTRACTION MAP")
aggregate_exp3, params_exp3 = construct_stochastic_attraction_map(
    policy_type='exp3',
    payoff_matrix_p1=game.payoff_matrix_p1,
    payoff_matrix_p2=game.payoff_matrix_p2,
    run_dir=RUN_DIR,
    grid_resolution=GRID_RESOLUTION,
    rounds_per_block=ROUNDS_PER_BLOCK,
    param_range=PARAM_RANGES['exp3'],
    horizon_schedule=HORIZON_SCHEDULE,
    min_replicates=MIN_REPLICATES,
    max_replicates=MAX_REPLICATES,
    replicate_batch_size=REPLICATE_BATCH_SIZE,
    modal_probability_stop=MODAL_PROBABILITY_STOP,
    policy_kwargs=POLICY_KWARGS['exp3'],
    base_seed=BASE_SEED,
)
plot_stage0_attraction_probability(
    aggregate_exp3['outcome_probability_maps'][0], params_exp3,
    'Repaired Anytime Exp3', 'stage-0-to-stage-2 weight shift (alpha)',
    save_path=RUN_DIR / 'figures' / 'exp3_stage0_attraction.png',
)
plot_modal_outcome_map(
    aggregate_exp3['modal_outcome_map'], aggregate_exp3['outcome_entropy_map'],
    params_exp3, 'Repaired Anytime Exp3', 'stage-0-to-stage-2 weight shift (alpha)',
    save_path=RUN_DIR / 'figures' / 'exp3_modal_outcome.png',
)
plot_scalar_grid(
    aggregate_exp3['final_horizon_map'], params_exp3,
    'Repaired Anytime Exp3', 'stage-0-to-stage-2 weight shift (alpha)', 'Adaptive Final Horizon', 'Rounds',
    save_path=RUN_DIR / 'figures' / 'exp3_final_horizon.png', cmap='magma',
)


In [ ]:
# =============================================================================
# CELL 27: DESCRIPTIVE STOCHASTIC-ATTRACTION COMPARISON
# =============================================================================
print('\n' + '=' * 96)
print('REPLICATED STOCHASTIC ATTRACTION TO (0,0)')
print('=' * 96)
policy_aggregates = {
    'Epsilon-Greedy': aggregate_epsilon_greedy,
    'Thompson Sampling': aggregate_thompson,
    'Repaired Anytime Exp3': aggregate_exp3,
}
for name, aggregate in policy_aggregates.items():
    p0 = aggregate['outcome_probability_maps'][0]
    print(
        f"{name:<28} mean P[(0,0)]={np.nanmean(p0):.3f} | "
        f"cells with P>=0.90: {100*np.mean(p0 >= 0.90):5.1f}% | "
        f"mean replicates: {np.mean(aggregate['replicate_count_map']):.1f} | "
        f"mean horizon: {np.mean(aggregate['final_horizon_map']):,.0f}"
    )
print('These are empirical attraction probabilities over policy-specific initial-bias slices.')
print('=' * 96)


In [ ]:
# =============================================================================
# CELL 28: TRAJECTORY ENSEMBLE CONFIGURATION
# =============================================================================
print("TRAJECTORY ENSEMBLE ANALYSIS")
TRAJ_N_BLOCKS = 500
TRAJ_ROUNDS_PER_BLOCK = 1000
TRAJ_N_RUNS = 50
TRAJ_CHECKPOINT = 200
print(f"{TRAJ_N_RUNS} trajectories; {TRAJ_N_BLOCKS} blocks x {TRAJ_ROUNDS_PER_BLOCK} rounds")
print("Agents are initialized once per trajectory and never reset between blocks.")


In [ ]:
# =============================================================================
# CELL 28: RUN ENSEMBLE - NEUTRAL INITIALIZATION
# =============================================================================
print("\n--- Neutral Initialization (β₁=0, β₂=0) ---")

ensemble_neutral = simulate_trajectory_ensemble(
    policy_type='epsilon_greedy',
    bias_p1=0.0,
    bias_p2=0.0,
    payoff_matrix_p1=game.payoff_matrix_p1,
    payoff_matrix_p2=game.payoff_matrix_p2,
    n_blocks=TRAJ_N_BLOCKS,
    rounds_per_block=TRAJ_ROUNDS_PER_BLOCK,
    n_runs=TRAJ_N_RUNS,
    checkpoint_interval=TRAJ_CHECKPOINT,
    policy_kwargs=POLICY_KWARGS['epsilon_greedy'],
    base_seed=BASE_SEED
)

plot_complete_ensemble(ensemble_neutral, 'Epsilon-Greedy (Neutral)',
                       save_path=RUN_DIR / 'figures' / 'ensemble_neutral_centipede.png')


In [ ]:
# =============================================================================
# CELL 29: RUN ENSEMBLE - COOPERATIVE BIAS
# =============================================================================
print("\n--- Both Optimistic about Cooperation (β₁=1, β₂=1) ---")

ensemble_coop = simulate_trajectory_ensemble(
    policy_type='epsilon_greedy',
    bias_p1=1.0,
    bias_p2=1.0,
    payoff_matrix_p1=game.payoff_matrix_p1,
    payoff_matrix_p2=game.payoff_matrix_p2,
    n_blocks=TRAJ_N_BLOCKS,
    rounds_per_block=TRAJ_ROUNDS_PER_BLOCK,
    n_runs=TRAJ_N_RUNS,
    checkpoint_interval=TRAJ_CHECKPOINT,
    policy_kwargs=POLICY_KWARGS['epsilon_greedy'],
    base_seed=BASE_SEED
)

plot_complete_ensemble(ensemble_coop, 'Epsilon-Greedy (Cooperative Bias)',
                       save_path=RUN_DIR / 'figures' / 'ensemble_cooperative_centipede.png')


In [ ]:
# =============================================================================
# CELL 30: RUN ENSEMBLE - ASYMMETRIC BIAS
# =============================================================================
print("\n--- Asymmetric (β₁=1, β₂=-1) ---")

ensemble_asym = simulate_trajectory_ensemble(
    policy_type='epsilon_greedy',
    bias_p1=1.0,
    bias_p2=-1.0,
    payoff_matrix_p1=game.payoff_matrix_p1,
    payoff_matrix_p2=game.payoff_matrix_p2,
    n_blocks=TRAJ_N_BLOCKS,
    rounds_per_block=TRAJ_ROUNDS_PER_BLOCK,
    n_runs=TRAJ_N_RUNS,
    checkpoint_interval=TRAJ_CHECKPOINT,
    policy_kwargs=POLICY_KWARGS['epsilon_greedy'],
    base_seed=BASE_SEED
)

plot_complete_ensemble(ensemble_asym, 'Epsilon-Greedy (Asymmetric)',
                       save_path=RUN_DIR / 'figures' / 'ensemble_asymmetric_centipede.png')


In [ ]:
# =============================================================================
# CELL 32: AGGREGATE TABLES, CHECKSUMS, AND RUN CLOSEOUT
# =============================================================================
def atomic_write_csv(path, fieldnames, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', newline='', encoding='utf-8', dir=path.parent,
                                     prefix=path.name, suffix='.tmp', delete=False) as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
        temporary = Path(stream.name)
    os.replace(temporary, path)


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


policy_specs = {
    'epsilon_greedy': (aggregate_epsilon_greedy, params_epsilon_greedy),
    'thompson': (aggregate_thompson, params_thompson),
    'exp3': (aggregate_exp3, params_exp3),
}
grid_rows = []
replicate_rows = []
combined_payload = {'metadata': {'run_id': RUN_ID, 'schema_version': 'centipede.stochastic-attraction.v3'}}

for policy, (aggregate, param_values) in policy_specs.items():
    for key, value in aggregate.items():
        combined_payload[f'{policy}__{key}'] = value
    combined_payload[f'{policy}__param_values'] = param_values
    for i, bias_p1 in enumerate(param_values):
        for j, bias_p2 in enumerate(param_values):
            probabilities = aggregate['outcome_probability_maps'][:, i, j]
            grid_rows.append({
                'policy': policy, 'grid_i': i, 'grid_j': j,
                'bias_p1': bias_p1, 'bias_p2': bias_p2,
                'p_stage0': probabilities[0], 'p_stage1': probabilities[1],
                'p_stage2': probabilities[2], 'p_mixed': probabilities[3],
                'modal_outcome': aggregate['modal_outcome_map'][i, j],
                'outcome_entropy': aggregate['outcome_entropy_map'][i, j],
                'n_replicates': aggregate['replicate_count_map'][i, j],
                'final_horizon': aggregate['final_horizon_map'][i, j],
                'stable_fraction': aggregate['stable_fraction_map'][i, j],
                'mean_first_dominance_time': aggregate['mean_first_dominance_map'][i, j],
            })
            result = load_grid_result(
                RUN_DIR / 'cells' / policy / f'cell_{i:02d}_{j:02d}.npz'
            )
            for r in range(result['metadata']['n_replicates']):
                terminal_counts = result['terminal_joint_counts'][r]
                terminal_total = int(terminal_counts.sum())
                replicate_rows.append({
                    'policy': policy, 'grid_i': i, 'grid_j': j,
                    'bias_p1': bias_p1, 'bias_p2': bias_p2,
                    'replicate_index': int(result['replicate_indices'][r]),
                    'replicate_seed': int(result['replicate_seeds'][r]),
                    'final_horizon': result['metadata']['final_horizon'],
                    'outcome_id': int(result['outcome_ids'][r]),
                    'stable': bool(result['stable_flags'][r]),
                    'terminal_stage0_share': terminal_counts[0, 0] / terminal_total,
                    'terminal_stage1_share': terminal_counts[1, 1] / terminal_total,
                    'terminal_stage2_share': terminal_counts[2, 2] / terminal_total,
                    'terminal_off_diagonal_share': 1.0 - np.trace(terminal_counts) / terminal_total,
                    'first_dominance_time': int(result['first_dominance_times'][r]),
                    'last_dominance_exit': int(result['last_dominance_exits'][r]),
                    'dominance_entries': int(result['dominance_entries'][r]),
                    'dominance_exits': int(result['dominance_exits'][r]),
                })

atomic_write_npz(RUN_DIR / 'aggregate' / 'basin_arrays.npz', combined_payload)
atomic_write_csv(RUN_DIR / 'aggregate' / 'grid_summary.csv', list(grid_rows[0]), grid_rows)
atomic_write_csv(RUN_DIR / 'aggregate' / 'replicate_summary.csv', list(replicate_rows[0]), replicate_rows)

manifest_path = RUN_DIR / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
manifest.update({
    'status': 'complete',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'game': {
        'name': 'unit-translated Smead finite-N quasi-centipede',
        'payoff_matrix_p1': game.payoff_matrix_p1.tolist(),
        'payoff_matrix_p2': game.payoff_matrix_p2.tolist(),
    },
    'python_version': platform.python_version(),
    'package_versions': {
        name: (package_version(name) if name != 'python' else platform.python_version())
        for name in ('numpy', 'matplotlib', 'seaborn', 'ray')
    },
})
output_files = [path for path in RUN_DIR.rglob('*') if path.is_file() and path != manifest_path]
manifest['output_checksums'] = {
    str(path.relative_to(RUN_DIR)): file_sha256(path) for path in output_files
}
atomic_write_json(manifest_path, manifest)
print(f"Completed run package: {RUN_DIR.resolve()}")


In [ ]:
# =============================================================================
# CELL 33: RAY SHUTDOWN
# =============================================================================
print("Shutting down Ray...")
ray.shutdown()
print("Finite-horizon initial-bias comparison complete.")
print("Future outputs use terminal_outcome_map_* and first_dominance_time_* names.")
print("The result pickle is timestamped; no mutable latest alias is written.")


In [ ]:
# =============================================================================
# CELL 34: INTERPRETIVE BOUNDARY
# =============================================================================
print(r"""
These computations estimate finite-horizon stochastic attraction probabilities
over explicit two-dimensional initial-bias slices. A cell's probability is the
share of independent replicated trajectories whose terminal quarter concentrates
on an outcome. Adaptive stopping extends the same agents without reset and records
the final horizon used at every cell.

The maps do not cover every possible policy state and do not establish an
asymptotic theorem. They show how sensitive the finite-horizon implementations
are to specified initial biases and private randomization.
""")
